# Week 2 - Preprocessing, part 2

# 1. Lesson: None

# 2. Weekly graph question

The Storytelling With Data book mentions planning on a "Who, What, and How" for your data story.  Write down a possible Who, What, and How for your data, using the ideas in the book.

## Who
- The audience is members of the **DX699 O2 cohort**, who are expected to have a basic understanding of machine learning and be more technical than a layperson.

## What
- The goal is to help the audience assess how **usable** this dataset is based on the exploratory data analysis (EDA).
- By reviewing the results, readers should be able to easily conclude whether the selected dataset is suitable for further analysis or modeling.

## How
- I will use the results of the exploratory data analysis to reach this conclusion by evaluating the dataset against the following criteria:
  - Inconsistent data
  - Duplicate rows
  - Missing values
  - Outliers
  - Inconsistent formatting
  - Categorical variables
  - Imbalanced classes
  - Data type inconsistencies


# 3. Homework - work with your own data

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

This week, you will do the same types of exercises as last week, but you should use your chosen datasets that someone in your class found last semester. (They likely will not be the particular datasets that you found yourself.)

### Here are some types of analysis you can do  Use Google, documentation, and ChatGPT to help you:

- Summarize the datasets using info() and describe()

- Are there any duplicate rows?

- Are there any duplicate values in a given column (when this would be inappropriate?)

- What are the mean, median, and mode of each column?

- Are there any missing or null values?

    - Do you want to fill in the missing value with a mean value?  A value of your choice?  Remove that row?

- Identify any other inconsistent data (e.g. someone seems to be taking an action before they are born.)

- Encode any categorical variables (e.g. with one-hot encoding.)

### Conclusions:

- Are the data usable?  If not, find some new data!

- Do you need to modify or correct the data in some way?

- Is there any class imbalance?  (Categories that have many more items than other categories).

In [ ]:
# Reading data from a CSV file
import kagglehub
from kagglehub import KaggleDatasetAdapter
file_path = "GoogleAds_DataAnalytics_Sales_Uncleaned.csv"
google_data = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "nayakganesh007/google-ads-sales-dataset",
  file_path
)
google_data.info()
google_data.describe()

General findings
- The types for the columns are a mix of objects and floats
- There are 13 collumns 
- There are 2600 rows


Inconsistent Data

In [ ]:
# Verify conversion rates
google_data['Clicks'] = pd.to_numeric(google_data['Clicks'], errors='coerce')
google_data['Conversions'] = pd.to_numeric(google_data['Conversions'], errors='coerce')
google_data['Conversion Rate'] = pd.to_numeric(google_data['Conversion Rate'], errors='coerce')

# Step 1: Calculate Conversion Rate (rounded to 3 decimals)
calculated_rate = (google_data['Conversions'] / google_data['Clicks']).round(3)

# Step 2: Flag mismatches
mismatch_flag = (google_data['Conversion Rate'] != 0) & \
                (~google_data['Conversion Rate'].isna()) & \
                (google_data['Conversion Rate'].round(3) != calculated_rate)

# Step 3: Extract bad rows with correct Conversion Rate for reference
bad_rows = google_data[mismatch_flag].copy()
bad_rows['Calculated Conversion Rate'] = calculated_rate[mismatch_flag]

# Step 4: Output summary
print("Total rows:", google_data.shape[0])
print("Number of rows with incorrect Conversion Rate:", bad_rows.shape[0])

# Step 5: Show a sample for EDA
bad_rows[['Ad_ID','Conversions','Clicks','Conversion Rate','Calculated Conversion Rate']].head(10)

# AD_ID should be a unique identifier
google_data['Ad_ID'] = google_data['Ad_ID'].astype(str)
unique_ad_ids = google_data['Ad_ID'].nunique()
if unique_ad_ids == google_data.shape[0]:
    print("Ad_ID is a unique identifier.")
else:
    print("Ad_ID is NOT a unique identifier. Not unique IDs:", google_data.shape[0] - unique_ad_ids)

# Checking for inconsistencies in campign names
google_data['Campaign_Name'] = google_data['Campaign_Name'].astype(str)
unique_campaigns = google_data['Campaign_Name'].nunique()
print("Total unique Campaign Names:", unique_campaigns)
google_data_incorrect_campigns = google_data['Campaign_Name'] != "Data Analytics Course"
if google_data_incorrect_campigns.sum() > 0:
    print("Inconsistent Campaign Names found:", google_data_incorrect_campigns.sum())
    print(google_data[google_data_incorrect_campigns]['Campaign_Name'].unique())
else:
    print("All Campaign Names are consistent.")

# Checking sales and costs for non dollar values
mismatch_flag_dollar_sales = (~google_data['Sale_Amount'].isna())
non_na_frame = google_data[mismatch_flag_dollar_sales].copy()
non_dollar_sales = non_na_frame['Sale_Amount'].str.contains(r'\$', na=False)

mismatch_flag_dollar_costs = (~google_data['Cost'].isna())
non_dollar_costs = google_data[mismatch_flag_dollar_costs]['Cost'].str.contains(r'\$', na=False)

if non_dollar_sales.sum() > 0:
    print("Non-dollar values found in Sale_Amount:", non_dollar_sales.shape[0] - non_dollar_sales.sum())
if non_dollar_costs.sum() > 0:
    print("Non-dollar values found in Cost:", non_dollar_costs.shape[0] - non_dollar_costs.sum())




Duplicate Rows

In [ ]:
get_duplicate_rows = google_data.duplicated()
get_duplicate_rows.sum()


## Missing Values

There are missing values in the following columns

In [ ]:
na_columns = google_data.columns[google_data.isna().any()].tolist()
for column in na_columns:
    na_count = google_data[column].isna().sum()
    print(f"Column '{column}' has {na_count} missing values.")

Outliers

In [ ]:
# Get numeric columns
from scipy.stats import zscore
numeric_columns = google_data.select_dtypes(include=[np.number]).columns.tolist()

for column in numeric_columns:
    mean_value = google_data[column].mean()      
    mode_value = google_data[column].mode()[0]   
    median_value = google_data[column].median()  
    print(f"Column '{column}': Mean = {mean_value}, Mode = {mode_value}, Median = {median_value}")
    # Defining my quantiles
    Q1 = google_data[column].quantile(0.25)
    Q3 = google_data[column].quantile(0.75)
    IQR = Q3 - Q1
    outliers = google_data[(google_data[column] < (Q1 - 1.5 * IQR)) | (google_data[column] > (Q3 + 1.5 * IQR))]
    print(f"Column '{column}': Number of outliers = {outliers.shape[0]}")
    # Z-score method
    z_scores = zscore(google_data[column], nan_policy='omit')
    outlier_threshold = 3
    z_outliers = google_data[(np.abs(z_scores) > outlier_threshold)]
    print(f"Column '{column}': Number of outliers (Z-score method) = {z_outliers.shape[0]}")


Inconsistent formating

In [ ]:
# Checking for inconsistencies in date formats
mismatch_flag_non_na = (~google_data['Ad_Date'].isna()) 
date_non_na = google_data[mismatch_flag_non_na].copy()
yyyy_mm_dd = date_non_na['Ad_Date'].str.match(r'^\d{4}/\d{2}/\d{2}$')
dd_mm_yyyy = date_non_na['Ad_Date'].str.match(r'^\d{2}-\d{2}-\d{4}$')
dash_yyyy_mm_dd = date_non_na['Ad_Date'].str.match(r'^\d{4}-\d{2}-\d{2}$')  # sometimes with dashes

# Flag rows that don’t match any expected format
bad_date_rows = date_non_na[~(yyyy_mm_dd | dd_mm_yyyy | dash_yyyy_mm_dd)]
print(f"Dates with incorrect format: {bad_date_rows.shape[0]} rows")

Categorical Variables

In [ ]:
# Get non-numeric columns
non_numeric_columns = google_data.select_dtypes(exclude=[np.number]).columns.tolist()

# Remove columns that are IDs, dates, or numeric-looking but stored as string
categorical_features = [col for col in non_numeric_columns 
                       if col not in ['Ad_ID', 'Ad_Date', 'Cost', 'Sale_Amount']]
google_data_encoded = pd.get_dummies(google_data, columns=categorical_features, drop_first=False)

# Check result
google_data_encoded.head()

Imbalanced Clases

Location and campiegn name while they do contain some data to be cleaned, they are composed of one value when cleaned and serve little to no purpose when trying to model anything with this dataset.

Data Type inconsistencies

In [ ]:
non_na_values = google_data.dropna()
for col in non_na_values.columns:
    types_in_column = non_na_values[col].map(type).unique()
    if len(types_in_column) == 1:
        print(f"All non-NA values in '{col}' are of the same type: {types_in_column[0]}")
    else:
        print(f"Non-NA values in '{col}' have multiple types: {types_in_column}")

1) There are not any duplicate rows in the dataset
2) There are missing values in the dataset, I think the best way to remediate is to remove them. Adding the mean my skew my conclusion
3) there were a few inconsistencies noticed. The conversion rate was wrong for a few observations. The campaign name was inconsistent, the date as well and the device names. 
4) The campaign name column was all one result, so was the location. Those columns are basically unusable
4) The data is usable, but it does require cleaning.

# 4. Storytelling With Data graph

Just like last week: choose any graph in the Introduction of Storytelling With Data. Use matplotlib to reproduce it in a rough way. I don't expect you to spend an enormous amount of time on this; I understand that you likely will not have time to re-create every feature of the graph. However, if you're excited about learning to use matplotlib, this is a good way to do that. You don't have to duplicate the exact values on the graph; just the same rough shape will be enough.  If you don't feel comfortable using matplotlib yet, do the best you can and write down what you tried or what Google searches you did to find the answers.

In [ ]:
from matplotlib import pyplot as plt
non_na = google_data[['Impressions','Clicks']].dropna()
non_na = non_na.sort_values(by='Impressions')

binned = non_na.copy()
binned['Imp_bin'] = pd.cut(non_na['Impressions'], bins=50)
agg = binned.groupby('Imp_bin')['Clicks'].mean().reset_index()
agg['Imp_bin'] = agg['Imp_bin'].cat.codes  # numeric x-axis

plt.figure(figsize=(8,5))
plt.plot(agg.index, agg['Clicks'], marker='o', color='blue')
plt.xlabel('Impressions (binned)')
plt.ylabel('Average Clicks')
plt.title('Avg Clicks vs Impressions (Binned)')
plt.grid(True)
plt.show()

I tried to recreate a line graph showed in the reading by comparing average clicks to binned impressions. However based on the result, there is little to no correlation between impressions and clicks. I googled "how to create a line graph in matplotlib"